# GDI Scoring Analysis: Current Network Concentration

This notebook analyzes the current network data to understand geographic concentration patterns and validate that our proposed scoring methodology improvements will accurately reflect risk.

**Goal**: Generate statistical summaries that show why current GDI scores (Ethereum 54, Polygon 52, Filecoin 48) read as "middling but okay" when underlying data shows alarming concentration.

## Analysis Steps
1. Load network data for Ethereum, Polygon, Filecoin
2. Calculate concentration metrics (top countries %, top orgs %)
3. Review current GDI/PDI/JDI/IHI scores
4. Identify concentration patterns that should be "alarming"
5. Propose threshold calibrations

In [2]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set up plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Define paths (we're in data/notebooks, so data dir is parent)
DATA_DIR = Path('..')
RAW_DIR = DATA_DIR / 'raw'
GDI_RESULTS = DATA_DIR / 'gdi_results.json'

## 1. Load Network Data

In [6]:
# Load raw network data
networks = {
    'ethereum': pd.read_csv(RAW_DIR / '2025-11-22-ethereum-ips.csv'),
    'polygon': pd.read_csv(RAW_DIR / '2025-11-22-polygon-ips.csv'),
    'filecoin': pd.read_csv(RAW_DIR / '2025-11-22-filecoin-ips.csv')
}

# Load current GDI scores
with open(GDI_RESULTS) as f:
    gdi_results = json.load(f)

# Display basic stats
print("Network Data Summary:")
print("=" * 60)
for name, df in networks.items():
    print(f"{name.upper()}:")
    print(f"  Total nodes: {len(df):,}")
    print(f"  Unique countries: {df['country'].nunique()}")
    print(f"  Unique regions: {df['region_name'].nunique()}")
    print(f"  Unique organizations: {df['org'].nunique()}")
    print(f"  Unique ASNs: {df['asname'].nunique()}")
    print()

Network Data Summary:
ETHEREUM:
  Total nodes: 17,356
  Unique countries: 103
  Unique regions: 607
  Unique organizations: 2274
  Unique ASNs: 1416

POLYGON:
  Total nodes: 11,242
  Unique countries: 82
  Unique regions: 423
  Unique organizations: 1346
  Unique ASNs: 885

FILECOIN:
  Total nodes: 256
  Unique countries: 25
  Unique regions: 73
  Unique organizations: 116
  Unique ASNs: 92



In [4]:
networks['ethereum'].head()

,ip,country,country_code,city,region_name,lat,lon,isp,org,as_raw,asname,hosting,proxy,mobile
0,109.111.106.105,Andorra,AD,Andorra la Vella,Andorra la Vella,42.5055,1.5243,"ANDORRA TELECOM, S.A.U.","ANDORRA TELECOM, S.A.U",AS6752 Andorra Telecom,ANDORRA,f,f,f
1,46.172.237.128,Andorra,AD,Andorra la Vella,Andorra la Vella,42.5055,1.5243,"ANDORRA TELECOM, S.A.U.","ANDORRA TELECOM, S.A.U",AS6752 Andorra Telecom,ANDORRA,f,f,f
2,185.132.201.163,Andorra,AD,Santa Coloma,Andorra la Vella,42.5014,1.4985,Andorra Telecom SAU,Bitcanal UK,AS6752 Andorra Telecom,ANDORRA,f,f,f
3,204.155.30.2,Anguilla,AI,The Valley,The Valley,18.2148,-63.0574,Hosting Solution Ltd.,IT Hosting Group,AS14576 Hosting Solution Ltd.,HOSTING-SOLUTIONS,t,f,f
4,186.109.17.235,Argentina,AR,Belén de Escobar,Buenos Aires,-34.3460,-58.7945,Telecom Argentina S.A.,Apolo -Gold-Telecom-Per,AS7303 Telecom Argentina S.A.,Telecom Argentina S.A.,f,f,f


def analyze_country_concentration(df, network_name, top_n=10):
    """Analyze country concentration for a network."""
    country_counts = df['country'].value_counts()
    country_pcts = (country_counts / len(df) * 100).round(2)
    
    # Calculate HHI
    shares = country_counts / len(df)
    hhi = (shares ** 2).sum()
    
    # Build results using pandas operations
    results = {
        'network': network_name,
        'total_nodes': len(df),
        'num_countries': len(country_counts),
        'top_1_country': country_counts.index[0],
        'top_1_pct': country_pcts.iloc[0],
        'top_2_pct': country_pcts.head(2).sum(),
        'top_3_pct': country_pcts.head(3).sum(),
        'top_5_pct': country_pcts.head(5).sum(),
        'top_10_pct': country_pcts.head(top_n).sum(),
        'country_hhi': hhi,
        'top_countries': country_pcts.head(top_n).to_dict()
    }
    
    return results

# Analyze all networks - use dict comprehension
country_analysis = {name: analyze_country_concentration(df, name) 
                    for name, df in networks.items()}

# Display results
print("\nCountry Concentration Analysis")
print("=" * 80)
for name, results in country_analysis.items():
    print(f"\n{name.upper()}:")
    print(f"  Total nodes: {results['total_nodes']:,}")
    print(f"  Countries: {results['num_countries']}")
    print(f"  Top country: {results['top_1_country']} ({results['top_1_pct']:.1f}%)")
    print(f"  Top 2 countries: {results['top_2_pct']:.1f}%")
    print(f"  Top 3 countries: {results['top_3_pct']:.1f}%")
    print(f"  Top 5 countries: {results['top_5_pct']:.1f}%")
    print(f"  Top 10 countries: {results['top_10_pct']:.1f}%")
    print(f"  Country HHI: {results['country_hhi']:.4f}")

In [7]:
def analyze_country_concentration(df, network_name, top_n=10):
    """Analyze country concentration for a network."""
    country_counts = df['country'].value_counts()
    total_nodes = len(df)
    
    # Calculate percentages
    country_pcts = (country_counts / total_nodes * 100).round(2)
    
    # Key concentration metrics
    top_1_pct = country_pcts.iloc[0] if len(country_pcts) > 0 else 0
    top_2_pct = country_pcts.iloc[:2].sum() if len(country_pcts) >= 2 else top_1_pct
    top_3_pct = country_pcts.iloc[:3].sum() if len(country_pcts) >= 3 else top_2_pct
    top_5_pct = country_pcts.iloc[:5].sum() if len(country_pcts) >= 5 else top_3_pct
    top_10_pct = country_pcts.iloc[:top_n].sum() if len(country_pcts) >= top_n else country_pcts.sum()
    
    # Calculate HHI (Herfindahl-Hirschman Index)
    shares = country_counts / total_nodes
    hhi = (shares ** 2).sum()
    
    results = {
        'network': network_name,
        'total_nodes': total_nodes,
        'num_countries': len(country_counts),
        'top_1_country': country_counts.index[0],
        'top_1_pct': top_1_pct,
        'top_2_pct': top_2_pct,
        'top_3_pct': top_3_pct,
        'top_5_pct': top_5_pct,
        'top_10_pct': top_10_pct,
        'country_hhi': hhi,
        'top_countries': country_pcts.head(top_n).to_dict()
    }
    
    return results

# Analyze all networks
country_analysis = {}
for name, df in networks.items():
    country_analysis[name] = analyze_country_concentration(df, name)

# Display results
print("\nCountry Concentration Analysis")
print("=" * 80)
for name, results in country_analysis.items():
    print(f"\n{name.upper()}:")
    print(f"  Total nodes: {results['total_nodes']:,}")
    print(f"  Countries: {results['num_countries']}")
    print(f"  Top country: {results['top_1_country']} ({results['top_1_pct']:.1f}%)")
    print(f"  Top 2 countries: {results['top_2_pct']:.1f}%")
    print(f"  Top 3 countries: {results['top_3_pct']:.1f}%")
    print(f"  Top 5 countries: {results['top_5_pct']:.1f}%")
    print(f"  Top 10 countries: {results['top_10_pct']:.1f}%")
    print(f"  Country HHI: {results['country_hhi']:.4f}")


Country Concentration Analysis

ETHEREUM:
  Total nodes: 17,356
  Countries: 103
  Top country: United States (30.5%)
  Top 2 countries: 44.7%
  Top 3 countries: 50.1%
  Top 5 countries: 60.0%
  Top 10 countries: 76.9%
  Country HHI: 0.1294

POLYGON:
  Total nodes: 11,242
  Countries: 82
  Top country: United States (30.6%)
  Top 2 countries: 46.8%
  Top 3 countries: 57.2%
  Top 5 countries: 65.7%
  Top 10 countries: 82.2%
  Country HHI: 0.1418

FILECOIN:
  Total nodes: 256
  Countries: 25
  Top country: China (23.1%)
  Top 2 countries: 45.3%
  Top 3 countries: 59.0%
  Top 5 countries: 73.5%
  Top 10 countries: 86.3%
  Country HHI: 0.1378


In [ ]:
# Visualize top countries for each network
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, (name, results) in enumerate(country_analysis.items()):
    ax = axes[idx]
    top_countries = pd.Series(results['top_countries'])
    
    top_countries.plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title(f'{name.upper()}\nTop 10 Countries', fontsize=12, fontweight='bold')
    ax.set_xlabel('Country')
    ax.set_ylabel('Percentage of Nodes (%)')
    ax.tick_params(axis='x', rotation=45)
    
    # Add horizontal line at 33% (single country dominance threshold)
    ax.axhline(y=33, color='red', linestyle='--', alpha=0.5, label='33% threshold')
    ax.legend()

plt.tight_layout()
plt.show()

def analyze_org_concentration(df, network_name, top_n=10):
    """Analyze organization/hosting concentration for a network."""
    # Filter out empty org names
    df_clean = df[df['org'].notna() & (df['org'] != '')]
    
    org_counts = df_clean['org'].value_counts()
    org_pcts = (org_counts / len(df) * 100).round(2)
    
    # Calculate HHI
    shares = org_counts / len(df)
    hhi = (shares ** 2).sum()
    
    results = {
        'network': network_name,
        'total_nodes': len(df),
        'nodes_with_org': len(df_clean),
        'num_orgs': len(org_counts),
        'top_1_org': org_counts.index[0] if len(org_counts) > 0 else 'N/A',
        'top_1_pct': org_pcts.iloc[0] if len(org_pcts) > 0 else 0,
        'top_2_pct': org_pcts.head(2).sum(),
        'top_3_pct': org_pcts.head(3).sum(),
        'top_5_pct': org_pcts.head(5).sum(),
        'top_10_pct': org_pcts.head(top_n).sum(),
        'org_hhi': hhi,
        'top_orgs': org_pcts.head(top_n).to_dict()
    }
    
    return results

# Analyze all networks
org_analysis = {name: analyze_org_concentration(df, name) 
                for name, df in networks.items()}

# Display results
print("\nOrganization Concentration Analysis")
print("=" * 80)
for name, results in org_analysis.items():
    print(f"\n{name.upper()}:")
    print(f"  Total nodes: {results['total_nodes']:,}")
    print(f"  Organizations: {results['num_orgs']}")
    print(f"  Top org: {results['top_1_org']} ({results['top_1_pct']:.1f}%)")
    print(f"  Top 2 orgs: {results['top_2_pct']:.1f}%")
    print(f"  Top 3 orgs: {results['top_3_pct']:.1f}%")
    print(f"  Top 5 orgs: {results['top_5_pct']:.1f}%")
    print(f"  Top 10 orgs: {results['top_10_pct']:.1f}%")
    print(f"  Org HHI: {results['org_hhi']:.4f}")

In [ ]:
def analyze_org_concentration(df, network_name, top_n=10):
    """Analyze organization/hosting concentration for a network."""
    # Clean org data (remove empty strings)
    df_clean = df[df['org'].notna() & (df['org'] != '')].copy()
    
    org_counts = df_clean['org'].value_counts()
    total_nodes = len(df)
    total_with_org = len(df_clean)
    
    # Calculate percentages based on total nodes
    org_pcts = (org_counts / total_nodes * 100).round(2)
    
    # Key concentration metrics
    top_1_pct = org_pcts.iloc[0] if len(org_pcts) > 0 else 0
    top_2_pct = org_pcts.iloc[:2].sum() if len(org_pcts) >= 2 else top_1_pct
    top_3_pct = org_pcts.iloc[:3].sum() if len(org_pcts) >= 3 else top_2_pct
    top_5_pct = org_pcts.iloc[:5].sum() if len(org_pcts) >= 5 else top_3_pct
    top_10_pct = org_pcts.iloc[:top_n].sum() if len(org_pcts) >= top_n else org_pcts.sum()
    
    # Calculate HHI
    shares = org_counts / total_nodes
    hhi = (shares ** 2).sum()
    
    results = {
        'network': network_name,
        'total_nodes': total_nodes,
        'nodes_with_org': total_with_org,
        'num_orgs': len(org_counts),
        'top_1_org': org_counts.index[0] if len(org_counts) > 0 else 'N/A',
        'top_1_pct': top_1_pct,
        'top_2_pct': top_2_pct,
        'top_3_pct': top_3_pct,
        'top_5_pct': top_5_pct,
        'top_10_pct': top_10_pct,
        'org_hhi': hhi,
        'top_orgs': org_pcts.head(top_n).to_dict()
    }
    
    return results

# Analyze all networks
org_analysis = {}
for name, df in networks.items():
    org_analysis[name] = analyze_org_concentration(df, name)

# Display results
print("\nOrganization Concentration Analysis")
print("=" * 80)
for name, results in org_analysis.items():
    print(f"\n{name.upper()}:")
    print(f"  Total nodes: {results['total_nodes']:,}")
    print(f"  Organizations: {results['num_orgs']}")
    print(f"  Top org: {results['top_1_org']} ({results['top_1_pct']:.1f}%)")
    print(f"  Top 2 orgs: {results['top_2_pct']:.1f}%")
    print(f"  Top 3 orgs: {results['top_3_pct']:.1f}%")
    print(f"  Top 5 orgs: {results['top_5_pct']:.1f}%")
    print(f"  Top 10 orgs: {results['top_10_pct']:.1f}%")
    print(f"  Org HHI: {results['org_hhi']:.4f}")

In [ ]:
# Visualize top orgs for each network
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, (name, results) in enumerate(org_analysis.items()):
    ax = axes[idx]
    top_orgs = pd.Series(results['top_orgs'])
    
    # Truncate long org names for display
    top_orgs.index = [org[:30] + '...' if len(org) > 30 else org for org in top_orgs.index]
    
    top_orgs.plot(kind='bar', ax=ax, color='darkorange')
    ax.set_title(f'{name.upper()}\nTop 10 Organizations', fontsize=12, fontweight='bold')
    ax.set_xlabel('Organization')
    ax.set_ylabel('Percentage of Nodes (%)')
    ax.tick_params(axis='x', rotation=45)
    
    # Add horizontal line at 25% threshold
    ax.axhline(y=25, color='red', linestyle='--', alpha=0.5, label='25% threshold')
    ax.legend()

plt.tight_layout()
plt.show()

## 4. Current GDI Scores Review

Compare the concentration metrics we just calculated with the current GDI scoring.

In [ ]:
# Create comparison dataframe
comparison_data = []

for result in gdi_results:
    network_id = result['id']
    
    # Get our calculated metrics
    country_metrics = country_analysis.get(network_id, {})
    org_metrics = org_analysis.get(network_id, {})
    
    comparison_data.append({
        'Network': result['name'],
        'Node Count': result['nodeCount'],
        'PDI': result['pdi'],
        'JDI': result['jdi'],
        'IHI': result['ihi'],
        'Top Country %': country_metrics.get('top_1_pct', 0),
        'Top 2 Countries %': country_metrics.get('top_2_pct', 0),
        'Top 3 Countries %': country_metrics.get('top_3_pct', 0),
        'Top Org %': org_metrics.get('top_1_pct', 0),
        'Top 3 Orgs %': org_metrics.get('top_3_pct', 0),
        'Country HHI': result['countryHHI'],
        'Org HHI': result['orgHHI']
    })

comparison_df = pd.DataFrame(comparison_data)
print("\nCurrent GDI Scores vs Concentration Metrics")
print("=" * 100)
print(comparison_df.to_string(index=False))

## 5. Critical Concentration Flags

Identify which networks trigger concerning concentration thresholds.

In [ ]:
# Define critical thresholds
THRESHOLDS = {
    'single_country_critical': 33,  # Single country > 33%
    'top2_countries_critical': 50,   # Top 2 countries > 50%
    'top3_countries_critical': 66,   # Top 3 countries > 66%
    'single_org_critical': 25,       # Single org > 25%
    'top3_orgs_critical': 50,        # Top 3 orgs > 50%
}

def check_concentration_flags(network_name, country_metrics, org_metrics):
    """Check if a network triggers any critical concentration flags."""
    flags = {}
    
    # Country flags
    flags['single_country_critical'] = country_metrics.get('top_1_pct', 0) > THRESHOLDS['single_country_critical']
    flags['top2_countries_critical'] = country_metrics.get('top_2_pct', 0) > THRESHOLDS['top2_countries_critical']
    flags['top3_countries_critical'] = country_metrics.get('top_3_pct', 0) > THRESHOLDS['top3_countries_critical']
    
    # Org flags
    flags['single_org_critical'] = org_metrics.get('top_1_pct', 0) > THRESHOLDS['single_org_critical']
    flags['top3_orgs_critical'] = org_metrics.get('top_3_pct', 0) > THRESHOLDS['top3_orgs_critical']
    
    return flags

# Check all networks
print("\nCritical Concentration Flags")
print("=" * 80)
for name in networks.keys():
    flags = check_concentration_flags(
        name, 
        country_analysis.get(name, {}),
        org_analysis.get(name, {})
    )
    
    print(f"\n{name.upper()}:")
    for flag_name, triggered in flags.items():
        status = "🚨 TRIGGERED" if triggered else "✓ OK"
        print(f"  {flag_name}: {status}")
    
    # Summary
    num_triggered = sum(flags.values())
    if num_triggered > 0:
        print(f"  ⚠️  {num_triggered} critical threshold(s) exceeded")
    else:
        print(f"  ✓ No critical thresholds exceeded")

## 6. Score Distribution Analysis

Visualize how current scores compare to concentration reality.

In [ ]:
# Create visualization showing scores vs concentration
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# PDI vs Top Country Concentration
ax1 = axes[0, 0]
for _, row in comparison_df.iterrows():
    ax1.scatter(row['Top Country %'], row['PDI'], s=200, alpha=0.6)
    ax1.text(row['Top Country %'], row['PDI'], row['Network'], 
             ha='center', va='bottom', fontsize=9)
ax1.set_xlabel('Top Country Concentration (%)', fontsize=11)
ax1.set_ylabel('PDI Score', fontsize=11)
ax1.set_title('PDI vs Country Concentration', fontsize=12, fontweight='bold')
ax1.axvline(x=33, color='red', linestyle='--', alpha=0.3, label='33% threshold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# JDI vs Top 2 Countries
ax2 = axes[0, 1]
for _, row in comparison_df.iterrows():
    ax2.scatter(row['Top 2 Countries %'], row['JDI'], s=200, alpha=0.6)
    ax2.text(row['Top 2 Countries %'], row['JDI'], row['Network'], 
             ha='center', va='bottom', fontsize=9)
ax2.set_xlabel('Top 2 Countries Concentration (%)', fontsize=11)
ax2.set_ylabel('JDI Score', fontsize=11)
ax2.set_title('JDI vs Top 2 Countries', fontsize=12, fontweight='bold')
ax2.axvline(x=50, color='red', linestyle='--', alpha=0.3, label='50% threshold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# IHI vs Top Org Concentration
ax3 = axes[1, 0]
for _, row in comparison_df.iterrows():
    ax3.scatter(row['Top Org %'], row['IHI'], s=200, alpha=0.6)
    ax3.text(row['Top Org %'], row['IHI'], row['Network'], 
             ha='center', va='bottom', fontsize=9)
ax3.set_xlabel('Top Organization Concentration (%)', fontsize=11)
ax3.set_ylabel('IHI Score', fontsize=11)
ax3.set_title('IHI vs Organization Concentration', fontsize=12, fontweight='bold')
ax3.axvline(x=25, color='red', linestyle='--', alpha=0.3, label='25% threshold')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Composite scores
ax4 = axes[1, 1]
comparison_df['Avg Score'] = comparison_df[['PDI', 'JDI', 'IHI']].mean(axis=1)
comparison_df.plot(x='Network', y=['PDI', 'JDI', 'IHI', 'Avg Score'], 
                   kind='bar', ax=ax4, width=0.8)
ax4.set_ylabel('Score', fontsize=11)
ax4.set_title('Current Sub-Index Scores', fontsize=12, fontweight='bold')
ax4.axhline(y=50, color='orange', linestyle='--', alpha=0.3, label='"Middling" score')
ax4.legend()
ax4.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 7. Summary Statistics Table

Generate a comprehensive summary table for the methodology documentation.

In [ ]:
# Create comprehensive summary
summary_data = []

for name, df in networks.items():
    country_m = country_analysis[name]
    org_m = org_analysis[name]
    
    # Find matching GDI result
    gdi_result = next((r for r in gdi_results if r['id'] == name), None)
    
    summary_data.append({
        'Network': name.upper(),
        'Nodes': f"{country_m['total_nodes']:,}",
        'Countries': country_m['num_countries'],
        'Organizations': org_m['num_orgs'],
        'Top Country': f"{country_m['top_1_country']} ({country_m['top_1_pct']:.1f}%)",
        'Top 2 Countries': f"{country_m['top_2_pct']:.1f}%",
        'Top 3 Countries': f"{country_m['top_3_pct']:.1f}%",
        'Top Organization': f"{org_m['top_1_org'][:30]} ({org_m['top_1_pct']:.1f}%)",
        'Top 3 Orgs': f"{org_m['top_3_pct']:.1f}%",
        'PDI': gdi_result['pdi'] if gdi_result else 'N/A',
        'JDI': gdi_result['jdi'] if gdi_result else 'N/A',
        'IHI': gdi_result['ihi'] if gdi_result else 'N/A',
    })

summary_df = pd.DataFrame(summary_data)

print("\n" + "=" * 120)
print("COMPREHENSIVE NETWORK CONCENTRATION SUMMARY")
print("=" * 120)
print(summary_df.to_string(index=False))
print("=" * 120)

## 8. Proposed Threshold Calibration

Based on the concentration patterns observed, propose new interpretation thresholds.

In [ ]:
print("\nPROPOSED THRESHOLD CALIBRATION")
print("=" * 80)

print("\n📊 OBSERVATIONS:")
print("\n1. All networks show significant concentration:")
for name in networks.keys():
    country_m = country_analysis[name]
    org_m = org_analysis[name]
    print(f"   • {name.upper()}: Top country {country_m['top_1_pct']:.1f}%, Top org {org_m['top_1_pct']:.1f}%")

print("\n2. Current scores (50-60 range) read as 'middling' but should signal risk")
print("\n3. Multiple critical thresholds exceeded across all networks")

print("\n\n🎯 PROPOSED NEW THRESHOLDS:")
print("""
Current interpretation (0-100 scale):
  0-33:  Poor/Centralized
  34-66: Moderate/Acceptable  ← All current networks fall here
  67-100: Good/Decentralized

Proposed interpretation (0-100 scale):
  0-40:  Critical Risk (severe concentration)
  41-60: High Risk (significant concentration)  ← Most networks should fall here
  61-75: Moderate Risk (some concentration)
  76-90: Low Risk (well distributed)
  91-100: Excellent (highly decentralized)

Rationale:
  - Current 50-60 scores should map to "High Risk" category
  - Scores should reflect that geographic concentration IS a risk
  - Very few (if any) real networks should score >75
  - Score of 90+ should be aspirational, not achievable by mediocre distribution
""")

print("\n🚨 CRITICAL CONCENTRATION FLAGS (Boolean indicators):")
print("""
  ⚠️  single_country_dominance: Top country > 33%
  ⚠️  top2_country_majority: Top 2 countries > 50%
  ⚠️  top3_country_supermajority: Top 3 countries > 66%
  ⚠️  single_org_dominance: Top organization > 25%
  ⚠️  top3_org_majority: Top 3 organizations > 50%
""")

## 9. Export Summary Data

Save the analysis results for use in documentation and further analysis.

In [ ]:
# Export summary to JSON
export_data = {
    'analysis_date': '2025-12-11',
    'networks_analyzed': list(networks.keys()),
    'concentration_summary': {
        name: {
            'country_metrics': country_analysis[name],
            'org_metrics': org_analysis[name],
            'critical_flags': check_concentration_flags(
                name,
                country_analysis[name],
                org_analysis[name]
            )
        }
        for name in networks.keys()
    },
    'proposed_thresholds': THRESHOLDS,
    'current_gdi_scores': gdi_results
}

# Save to file
output_path = DATA_DIR / 'concentration_analysis_summary.json'
with open(output_path, 'w') as f:
    json.dump(export_data, f, indent=2, default=str)

print(f"\n✅ Analysis summary exported to: {output_path}")
print(f"\n📊 Total networks analyzed: {len(networks)}")
print(f"📁 Output file size: {output_path.stat().st_size / 1024:.1f} KB")

## Conclusions

**Key Findings:**

1. **All networks show significant geographic concentration** that should be flagged as concerning
2. **Current scores (50-60 range) fall in the "moderate/acceptable" band** but underlying concentration is alarming
3. **Multiple critical thresholds are exceeded** across all networks
4. **Score interpretation ranges need recalibration** to accurately reflect risk

**Recommended Changes:**

1. **Recalibrate interpretation thresholds** so current networks fall in "High Risk" category
2. **Add boolean concentration flags** for critical patterns
3. **Equal-weight sub-indices** (PDI, JDI, IHI) instead of current weighting
4. **Implement hard floor caps** so critically low sub-index drags down composite score
5. **Remove network size from scoring** (keep as display-only metadata)

**Next Steps:**
- Implement new scoring formula (geobeat-ejx)
- Update threshold constants (geobeat-apq)
- Add boolean flags to output schema (geobeat-p5s)
- Regenerate all network data (geobeat-nzr)
- Verify frontend compatibility (geobeat-9jh)
- Update methodology documentation (geobeat-8ow)